In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

In [4]:
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (691369, 14)
Test shape: (296302, 13)


In [5]:
X = train.drop(columns=["addicted_label", "id"]).copy()
y = train["addicted_label"].copy()

X_test = test.drop(columns=["id"]).copy()

print("X:", X.shape)
print("y:", y.shape)
print("X_test:", X_test.shape)

X: (691369, 12)
y: (691369,)
X_test: (296302, 12)


In [6]:
cat_cols = [
    "gender",
    "stress_level",
    "academic_work_impact",
]

for col in cat_cols:
    X[col] = X[col].fillna("Missing")
    X_test[col] = X_test[col].fillna("Missing")

In [9]:
X_train, X_valid, y_train, y_valid = train_test_split(  # 训练特征，测试特征，训练标签，测试标签。固定顺序
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,  # 按照标签 y 里各类别的占比，按比例分到训练集和验证集，防止切分后某一类样本消失、比例失衡。分类任务（尤其是样本不均衡）要加
)

print("Train:", X_train.shape)
print("Valid:", X_valid.shape)

print()
print("Train target ratio:",y_train.value_counts(normalize=True),"\n")
print("Valid target ratio:", y_valid.value_counts(normalize=True),"\n")


Train: (553095, 12)
Valid: (138274, 12)

Train target ratio: addicted_label
1    0.709424
0    0.290576
Name: proportion, dtype: float64 

Valid target ratio: addicted_label
1    0.709425
0    0.290575
Name: proportion, dtype: float64 



In [8]:
model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=50,
    allow_writing_files=False,
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_cols,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=50,
)

0:	test: 0.9031016	best: 0.9031016 (0)	total: 163ms	remaining: 1m 21s
50:	test: 0.9291675	best: 0.9291675 (50)	total: 4.3s	remaining: 37.9s
100:	test: 0.9350226	best: 0.9350226 (100)	total: 8.69s	remaining: 34.3s
150:	test: 0.9384177	best: 0.9384177 (150)	total: 13.8s	remaining: 32s
200:	test: 0.9411362	best: 0.9411362 (200)	total: 18.2s	remaining: 27.1s
250:	test: 0.9433158	best: 0.9433158 (250)	total: 22.3s	remaining: 22.2s
300:	test: 0.9454591	best: 0.9454591 (300)	total: 26.4s	remaining: 17.5s
350:	test: 0.9470115	best: 0.9470115 (350)	total: 30.6s	remaining: 13s
400:	test: 0.9484022	best: 0.9484022 (400)	total: 35.1s	remaining: 8.66s
450:	test: 0.9497764	best: 0.9497764 (450)	total: 39.4s	remaining: 4.28s
499:	test: 0.9509171	best: 0.9509171 (499)	total: 44.5s	remaining: 0us

bestTest = 0.9509171423
bestIteration = 499



CatBoostClassifier(allow_writing_files=False, depth=6, eval_metric='AUC', iterations=500, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=50)

In [10]:
valid_pred = model.predict_proba(X_valid)[:, 1]

valid_auc = roc_auc_score(
    y_valid,
    valid_pred
)

print(f"Validation AUC: {valid_auc:.6f}")

Validation AUC: 0.950917


In [11]:
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_,
}).sort_values(
    "importance",
    ascending=False
)

display(feature_importance)

,feature,importance
1,daily_screen_time_hours,30.385165
8,weekend_screen_time,27.528522
2,social_media_hours,18.204339
7,app_opens_per_day,9.355938
6,notifications_per_day,8.380489
4,work_study_hours,2.784184
3,gaming_hours,1.922442
0,age,0.882247
5,sleep_hours,0.556674
9,gender,0.000000


In [12]:
best_iterations = model.get_best_iteration() + 1

print("Best iterations:", best_iterations)

Best iterations: 500


In [13]:
final_model = CatBoostClassifier(
    iterations=best_iterations,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=50,
    allow_writing_files=False,
)

final_model.fit(
    X,
    y,
    cat_features=cat_cols,
)

0:	total: 99.6ms	remaining: 49.7s
50:	total: 4.83s	remaining: 42.5s
100:	total: 9.82s	remaining: 38.8s
150:	total: 15s	remaining: 34.6s
200:	total: 20.1s	remaining: 29.9s
250:	total: 24.9s	remaining: 24.7s
300:	total: 29.9s	remaining: 19.8s
350:	total: 34.9s	remaining: 14.8s
400:	total: 39.6s	remaining: 9.79s
450:	total: 44.4s	remaining: 4.83s
499:	total: 49.1s	remaining: 0us


CatBoostClassifier(allow_writing_files=False, depth=6, eval_metric='AUC', iterations=500, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=50)

In [14]:
test_pred = final_model.predict_proba(X_test)[:, 1]

print(test_pred[:10])
print()
print("Min:", test_pred.min())
print("Max:", test_pred.max())
print("Mean:", test_pred.mean())

[0.99760025 0.91114701 0.96064097 0.97951233 0.99627315 0.80945235
 0.89952116 0.37776546 0.97295637 0.44671003]

Min: 0.0017004034841042914
Max: 0.9999913314126209
Mean: 0.7090278257274583


In [15]:
submission = sample_submission.copy()

submission["addicted_label"] = test_pred

display(submission.head())
print(submission.shape)
print(submission.isna().sum())

,id,addicted_label
0,691369,0.997600
1,691370,0.911147
2,691371,0.960641
3,691372,0.979512
4,691373,0.996273


(296302, 2)
id                0
addicted_label    0
dtype: int64


In [16]:
SUBMISSION_DIR = PROJECT_ROOT / "submissions"
SUBMISSION_DIR.mkdir(exist_ok=True)

submission_path = (
    SUBMISSION_DIR /
    "catboost_baseline_v1.csv"
)

submission.to_csv(
    submission_path,
    index=False,
)

print("Saved to:", submission_path)

Saved to: /Users/c.c./Developer/Competitions/predicting-smartphone-addiction/submissions/catboost_baseline_v1.csv
